# Notebook about the research on LiDi 

LiDi is a singlet fission molecule who consist of two pentacene chromophores connected by a linear diacetylene spacer.   
We are focusing on the ground state dynamics and specifically, in the rotation around the linker. This process is beign tracked by measuring the dihedral angle between the two pentacene planes.


In [ ]:
from IPython.display import Image
Image(filename='/hpchome/dlp6/Lidiproject/scan/asmusconstraints/ff_fit/pbe0/lidi3-1.png', width=800, height=600) 

## 1-Geometry optimization of LiDi

The geometry optimization provides a minimum around 14 degrees

## 2-Constrained scan around dihedral angle

We create the inputs for the scan. We rotated one of the pentacene planes plus the side group with it, we do a rotation of 360º with a step of 10º.  
Also there, there is a function called _{create_input}_ which creates the inputs for all the single point ORCA calculations in DFT with an specific density functional for each dihedral angle.

In <span style="color:cyan"> spfullconstrained_solvated_360 or functionals </span> there is the script to send all the ORCA inputs at once

In [ ]:
def create_input(dih):
    inp = f'''\
! BLYP RIJCOSX def2-TZVP D4 verytightscf def2/J KDIIS MiniPrint CPCM(THF) NoAutoStart
#dihedral: {dih:4.2f}
%pal
    nprocs 64
end

%maxcore 3900

'''
    return inp

import os
import numpy as np
from ase.io import read, write
from ase import Atoms
from ase.constraints import FixAtoms

opth = '/hpchome/dlp6/Lidiproject/scan/asmusconstraints'
atoms0 = read('tlidiopt4.xyz')
atoms = read('tlidiopt4.xyz')
# Definición del dihedro
dih_idx = [31, 28, 6, 2]
#dih_idx = [2, 6, 28, 31]

# Índices como en tu script original

fix_idx1 = [atom.index for atom in atoms if atom.symbol == 'C' and atom.index in dih_idx]

tetra2_idx = list(range(0, 22))
fix_idx22 = [atom.index for atom in atoms if atom.symbol == 'C' and atom.index in tetra2_idx] # Derecha C

tetra3_idx = list(range(50, 64))
fix_idx32 = [atom.index for atom in atoms if atom.symbol == 'H' and atom.index in tetra3_idx] # Derecha H 

tetra2_idx = list(range(22, 44))
fix_idx23 = [atom.index for atom in atoms if atom.symbol == 'C' and atom.index in tetra2_idx] # Izquierda C

tetra3_idx = list(range(64, 76))
fix_idx33 = [atom.index for atom in atoms if atom.symbol == 'H' and atom.index in tetra3_idx] # Izquierda H

sidegroups2 = list(range(46,48)) + list(range(76, 77)) + list(range(82, 94)) + list(range(103, 121)) + list(range(139, 147)) + list(range(155, 156)) # Side group derecha

sidegroups3 = list(range(44, 46)) + list(range(77, 82)) + list(range(94, 103)) + list(range(121, 139)) + list(range(148, 155)) # Side group izquierda


rot_idx = fix_idx22 + fix_idx32 + sidegroups2 # Derecha rotados

fix_idx = fix_idx23 + fix_idx33 + sidegroups3 # Izquierda fijos

# Ángulos de scan
dihedral_angles = np.arange(0, 370, 10)


for i, atoms in enumerate(dihedral_angles):
    this_dih = dihedral_angles[i]
    inp = create_input(dih=this_dih)
    this_path = opth + '/functionals/blyp/solvated/'
    os.makedirs(this_path, exist_ok=True)
    with open(this_path + f'rr_s0_{i:02d}.inp', 'w')  as f:
        f.write(inp)

        f.write('%geom\nConstraints\n')
        for idx in fix_idx: 
            f.write(f'{{C {idx:d} C}}\n')
        f.write('end\nend\n\n')

        atoms = read(f'/hpchome/dlp6/Lidiproject/scan/asmusconstraints/dihedral_scan_right_360/rot_{i:03d}.xyz')
        
        pos = atoms.get_positions()
        atm = atoms.get_chemical_symbols()

        f.write('*xyz 0 1\n') # mult
        for i, row in enumerate(pos):
            f.write(f'{atm[i]:3s}'+''.join(f'{val:15.10f}' for val in row) + '\n')
        f.write('*')

## 3-Optimization of the force field for LiDi

With __Vicem__ software, we can create a GAFF force field using the topology and the coordinates from a random structure of LiDi.    
Then we perform the scan using GAFF and we compare with DFT, the difference is the correction to the potential energy. This correction is fitted by a Fourier torsion correction and its coefficients are linearly fitted giving us an absolute minimum

In [ ]:
import numpy as np
from pathlib import Path
from ase.io import read

import sys
sys.path.insert(0, '..')

from vicem import OpenMMCalculator, from_amber, reorder_atoms
from vicem.opt_ff import (
    FourierTorsionCorrection,
    NEBCostFunction,
    fit_correction,
)

from ase.constraints import FixInternals
from ase.optimize import BFGS

# Loading files and setting up paths

DATA_DIR = '.'

DIH_IDX = [31, 28, 6, 2]

ROOT = Path.cwd()
if not (ROOT / "vicem").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

LIDI_DIR = ROOT

PRMTOP = LIDI_DIR / "lidi_02_xtb_LDI_revB.prmtop"
INPCRD = LIDI_DIR / "lidi_02_xtb_LDI_revB.inpcrd"

TARGET_XYZ = LIDI_DIR / "lidisolvopt.xyz"
MEP_XYZ = LIDI_DIR / "neb-ci_perp-paral_MEP_trj.xyz"
CHARGES_XYZ = LIDI_DIR / "lidichargessolvpbe0.chelpg.xyz"
SCAN_XYZ = LIDI_DIR / 'scan_360.xyz'
SCAN_ENERGIES = LIDI_DIR / "energies_rel_pbe0_solvated.txt"

assert PRMTOP.exists(), PRMTOP
assert INPCRD.exists(), INPCRD
assert TARGET_XYZ.exists(), TARGET_XYZ
assert MEP_XYZ.exists(), MEP_XYZ
assert CHARGES_XYZ.exists(), CHARGES_XYZ
assert SCAN_XYZ.exists(), SCAN_XYZ
assert SCAN_ENERGIES.exists(), SCAN_ENERGIES

# Functions:

def make_calc():
    """
    Fresh unconstrained OpenMMCalculator from GAFF prmtop.
    """
    return OpenMMCalculator.from_amber(
        str(PRMTOP),
        str(INPCRD),
        nonbonded_method='NoCutoff',
        constraints=None,
    )


def get_perm():
    """
    Obtain atom permutation mapping between DFT xyz ordering
    and prmtop ordering.
    """

    calc = make_calc()

    reference_atoms = calc.get_atoms().copy()

    reference_centered = reference_atoms.copy()
    reference_centered.positions -= reference_centered.positions.mean(axis=0)

    # Use one DFT geometry as probe
    probe = read(str(MEP_XYZ), index=-1)

    probe_centered = probe.copy()
    probe_centered.positions -= probe_centered.positions.mean(axis=0)

    _, perm = reorder_atoms(
        probe_centered,
        reference_centered,
        element_map={"Si": "S"},
    )

    return perm


def make_calc_retargeted():
    """
    Create GAFF calculator with:
      - bonded equilibrium values updated from DFT geometry
      - CHELPG charges reordered into prmtop ordering

    Includes validation that:
      - CHARGES_XYZ and TARGET_XYZ share identical ordering
      - permutation is internally consistent
      - charges are assigned to the correct atoms
    """

    # Get permutation

    perm = get_perm()

    # Reference prmtop atoms

    calc = make_calc()

    reference_atoms = calc.get_atoms().copy()

    reference_symbols = np.array(
        reference_atoms.get_chemical_symbols()
    )

    # Read target geometry

    target_raw = read(str(TARGET_XYZ))

    # Reorder DFT geometry into prmtop ordering
    target_prmtop_order = target_raw[perm]

    # Force symbols to match topology
    target_prmtop_order.set_chemical_symbols(
        reference_atoms.get_chemical_symbols()
    )

    # Create retargeted calculator

    calc_retargeted = make_calc()

    calc_retargeted.update_bonded_equilibria_from_atoms(
        target_prmtop_order,
        update_constraints=False,
        update_positions=True,
    )

    # Read CHELPG charges xyz

    charges_atoms = read(str(CHARGES_XYZ))

    charges_raw = []
    symbols_raw = []

    with open(CHARGES_XYZ) as f:

        lines = f.readlines()[2:]

        for line in lines:

            fields = line.split()
            # first column = symbol
            symbols_raw.append(fields[0])

            # last column = charge
            charges_raw.append(float(fields[-1]))

    symbols_raw = np.array(symbols_raw)
    charges_raw = np.array(charges_raw)

    # Validate CHARGES_XYZ vs TARGET_XYZ ordering

    assert len(charges_atoms) == len(target_raw)

    geom_diff = np.linalg.norm(
        charges_atoms.positions
        - target_raw.positions,
        axis=1,
    )

    assert geom_diff.max() < 1e-5, (
        "CHARGES_XYZ and TARGET_XYZ "
        "have different atom orderings."
    )

    # Apply permutation

    symbols_perm = symbols_raw[perm]
    charges_perm = charges_raw[perm]

    # Consistent Si -> S mapping
    symbols_perm = np.where(
        symbols_perm == "Si",
        "S",
        symbols_perm,
    )

    # Update charges

    calc_retargeted.update_charges(
        range(len(charges_perm)),
        charges_perm,
    )

    return calc_retargeted


# Calculation begins

dft_mep = read(SCAN_XYZ, index=':')

dft_scan = np.loadtxt(
    SCAN_ENERGIES,
    delimiter='\t',
    usecols=(0, 1)
)

angles = dft_scan[:, 0]
energies_rel = dft_scan[:, 1]

dft_E_meV = energies_rel * 1000

# Load permutation

perm = get_perm()

# Ref atoms = OG prmtop
calc = make_calc()

ref_atoms = calc.get_atoms().copy()

ref_atoms.positions -= ref_atoms.positions.mean(0)

# Scan using modified GAFF

gaff_scan_E_eV = []
ase_dihedrals = []
gaff_relaxed_images = []

print('\nGAFF single points (constrained dihedral)')
print('-' * 60)

for i, dft_atoms in enumerate(dft_mep):

    # Apply permutation to 360 scan

    dft_atoms_reordered = dft_atoms[perm]

    # Ensure symbols match
    dft_atoms_reordered.set_chemical_symbols(
        ref_atoms.get_chemical_symbols()
    )

    # Image will be used for the calculation of the energy
    image = ref_atoms.copy()
    image.set_positions(
        dft_atoms_reordered.get_positions()
    )

    #Set the retargeted calculator for the GAFF-DFT scan
    image.calc = make_calc_retargeted()

    distances = image.get_all_distances(mic=False)

    min_dist = np.min(distances[distances > 0])

    #Sanity checks
    print(
        f'Image {i:2d} | '
        f'min distance = {min_dist:.3f} angstroms'
    )

    if min_dist < 0.5:
        print("WARNING: suspiciously short contact detected")

    ase_dih = image.get_dihedral(*DIH_IDX)

    ase_dihedrals.append(ase_dih)

    E = image.get_potential_energy()
    gaff_relaxed_images.append(image)
    gaff_scan_E_eV.append(E)

    print(
        f'  dih = {ase_dih:7.1f}°   '
        f'E = {E:.6f} eV'
    )


ase_dihedrals = np.array(ase_dihedrals)

gaff_scan_E_eV = np.array(gaff_scan_E_eV)

gaff_scan_E_meV = (
    gaff_scan_E_eV - gaff_scan_E_eV.min()
) * 1000

# Potential correction
V_corr_meV = dft_E_meV - gaff_scan_E_meV

with open("optFF.txt", "w") as f:

    for i in range(len(dft_mep)):

        f.write(
            f'\n'
            f'{i:4d} '
            f'{ase_dihedrals[i]:12.1f} '
            f'{dft_E_meV[i]:12.2f} '
            f'{gaff_scan_E_meV[i]:13.2f} '
            f'{V_corr_meV[i]:13.2f}'
        )

print(
    f'\n'
    f'{"Image":>6} '
    f'{"ASE dih (°)":>12} '
    f'{"E_DFT (meV)":>12} '
    f'{"E_GAFF (meV)":>13} '
    f'{"V_corr (meV)":>13}'
)

print('-' * 65)

for i in range(len(dft_mep)):

    print(
        f'{i:6d} '
        f'{ase_dihedrals[i]:12.1f} '
        f'{dft_E_meV[i]:12.2f} '
        f'{gaff_scan_E_meV[i]:13.2f} '
        f'{V_corr_meV[i]:13.2f}'
    )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

ase_dihedrals = np.loadtxt("filesforasmus/optFF.txt", usecols=1, skiprows=1)  # Ángulos de las imágenes según ASE (0–360°)
ase_dihedrals[-1] = 360
print(ase_dihedrals)
dft_E_meV = np.loadtxt("filesforasmus/optFF.txt", usecols=2, skiprows=1)  # Energías de DFT (meV)
dft_E_meV2 = dft_E_meV[:]
print(np.round(dft_E_meV2, 1))
gaff_scan_E_meV = np.loadtxt("filesforasmus/optFF.txt", usecols=3, skiprows=1)  # Energías de GAFF (meV)
# gaff_scan_E_meV2 = gaff_scan_E_meV * -1
# gaff_scan_E_meV2 -= gaff_scan_E_meV2.min()
gaff_scan_E_meV2 = gaff_scan_E_meV
print(gaff_scan_E_meV2)
V_corr_meV = np.loadtxt("filesforasmus/optFF.txt", usecols=4, skiprows=1)  # Corrección de energía (meV)
V_corr_meV2 = V_corr_meV[:]
print(V_corr_meV)

In [ ]:
# Fit Fourier correction (cosine-only, 2 harmonics)
correction = FourierTorsionCorrection(DIH_IDX, n_harmonics=2, cosine_only=True)
params_scan, rmse_scan = correction.fit_linear(ase_dihedrals, V_corr_meV2)

print(f'Linear fit RMSE: {rmse_scan:.2f} meV')
print(f'Parameters (meV):')
for name, val in zip(correction.param_names(), params_scan):
    print(f'  {name} = {val:+.4f}')

In [ ]:

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8))


# Upper: correction potential
label = f'Fourier fit ({params_scan[0]:.2f}, {params_scan[1]:.2f}, {params_scan[2]:.2f})'
ax1.plot(phi, V_fit, 'C0-', lw=2, label=label)
ax1.plot(ase_dihedrals, V_corr_meV, 'ko', ms=7, mfc='none', label='DFT − GAFF')
ax1.set_xlabel('Dihedral angle (deg)')
ax1.set_xlim(0, 360)
ax1.set_xticks(np.arange(0, 361, 30))
ax1.set_ylabel(f'V$_{{corr}}$ (meV)', fontsize=14)
ax1.text(0.05, 0.9, "(b)", transform=ax1.transAxes, fontsize=16,
        verticalalignment='top')
ax1.legend(loc='upper center')

# Lower: corrected scan vs DFT
scan_with_fit = gaff_scan_E_meV + correction.predict(ase_dihedrals, params_scan)
ax2.plot(ase_dihedrals, dft_E_meV, 'ko', ms=7, mfc='none', label='DFT (THF)')
ax2.plot(ase_dihedrals, gaff_scan_E_meV, 's-', color='C1', ms=5, label='GAFF-DFT')
ax2.plot(ase_dihedrals, scan_with_fit, '^-', color='C2', ms=5, label='GAFF-DFT+Corr.')
ax2.set_xlabel('Dihedral angle (deg)')
ax2.set_xlim(0, 360)
ax2.hlines(25, 0, 360, )
ax2.set_ylim(0,60)
ax2.set_xticks(np.arange(0, 361, 30))
ax2.set_ylabel('Relative energy (meV)', fontsize=14)
ax2.text(0.05, 0.9, "(c)", transform=ax2.transAxes, fontsize=16,
        verticalalignment='top')
ax2.legend()

fig.tight_layout()
plt.show()

## 4-MD simulations

To test the FF, we run 2 MD simulations (both in solvent and the correction included), a short NPT and a long NVT.  
The way to analyze it is to do a histogram sampling the dihedral angles along the trajectory and folding them between 0-90º

In [ ]:

import numpy as np
from ase.io import read
from vicem import OpenMMCalculator
import MDAnalysis as mda
from pathlib import Path
import sys
from vicem.calculator import OpenMMCalculator
from vicem import from_amber, reorder_atoms
from vicem.opt_ff import FourierTorsionCorrection

# Loading files and setting up paths

DATA_DIR = '.'

DIH_IDX = [31, 28, 6, 2]

ROOT = Path.cwd()
if not (ROOT / "vicem").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

LIDI_DIR = ROOT 

PRMTOP = LIDI_DIR / "lidi_02_xtb_LDI_revB.prmtop"
INPCRD = LIDI_DIR / "lidi_02_xtb_LDI_revB.inpcrd"

TARGET_XYZ = LIDI_DIR / "lidisolvopt.xyz"
MEP_XYZ = LIDI_DIR / "neb-ci_perp-paral_MEP_trj.xyz"
CHARGES_XYZ = LIDI_DIR / "lidichargessolvpbe0.chelpg.xyz"
SCAN_XYZ = LIDI_DIR / 'scan_360.xyz'
SCAN_ENERGIES = LIDI_DIR / "energies_rel_pbe0_solvated.txt"

assert PRMTOP.exists(), PRMTOP
assert INPCRD.exists(), INPCRD
assert TARGET_XYZ.exists(), TARGET_XYZ
assert MEP_XYZ.exists(), MEP_XYZ
assert CHARGES_XYZ.exists(), CHARGES_XYZ
assert SCAN_XYZ.exists(), SCAN_XYZ
assert SCAN_ENERGIES.exists(), SCAN_ENERGIES

# Functions: 

def measure_dihedral(atoms, i, j, k, l):
    """
    Dihedral angle in degrees (manual convention).

    Opposite sign convention to ASE/OpenMM.
    """
    p = atoms.get_positions()

    b1 = p[j] - p[i]
    b2 = p[k] - p[j]
    b3 = p[l] - p[k]

    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)

    n1 /= np.linalg.norm(n1)
    n2 /= np.linalg.norm(n2)

    m1 = np.cross(n1, b2 / np.linalg.norm(b2))

    return np.degrees(np.arctan2(np.dot(m1, n2), np.dot(n1, n2)))


def fix_wrap(dihs):
    """Fix ASE 0–360 wrapping."""
    return np.array([x - 360 if x > 350 else x for x in dihs])


def make_calc():
    """
    Fresh unconstrained OpenMMCalculator from GAFF prmtop.
    """
    return OpenMMCalculator.from_amber(
        str(PRMTOP),
        str(INPCRD),
        platform='CUDA',
        nonbonded_method='NoCutoff',
        constraints=None,
    )


def get_perm():
    """
    Obtain atom permutation mapping between DFT xyz ordering
    and prmtop ordering.
    """

    calc = make_calc()

    reference_atoms = calc.get_atoms().copy()

    reference_centered = reference_atoms.copy()
    reference_centered.positions -= reference_centered.positions.mean(axis=0)

    # Use one DFT geometry as probe
    probe = read(str(MEP_XYZ), index=-1)

    probe_centered = probe.copy()
    probe_centered.positions -= probe_centered.positions.mean(axis=0)

    _, perm = reorder_atoms(
        probe_centered,
        reference_centered,
        element_map={"Si": "S"},
    )

    return perm


def make_calc_retargeted():
    """
    Create GAFF calculator with:
      - bonded equilibrium values updated from DFT geometry
      - CHELPG charges reordered into prmtop ordering

    Includes validation that:
      - CHARGES_XYZ and TARGET_XYZ share identical ordering
      - permutation is internally consistent
      - charges are assigned to the correct atoms
    """

    # ============================================================
    # Get permutation
    # ============================================================

    perm = get_perm()

    # ============================================================
    # Reference prmtop atoms
    # ============================================================

    calc = make_calc()

    reference_atoms = calc.get_atoms().copy()

    reference_symbols = np.array(
        reference_atoms.get_chemical_symbols()
    )

    # ============================================================
    # Read target geometry
    # ============================================================

    target_raw = read(str(TARGET_XYZ))

    # Reorder DFT geometry into prmtop ordering
    target_prmtop_order = target_raw[perm]

    # Force symbols to match topology
    target_prmtop_order.set_chemical_symbols(
        reference_atoms.get_chemical_symbols()
    )

    # ============================================================
    # Create retargeted calculator
    # ============================================================

    calc_retargeted = make_calc()

    calc_retargeted.update_bonded_equilibria_from_atoms(
        target_prmtop_order,
        update_constraints=False,
        update_positions=True,
    )

    # ============================================================
    # Read charges xyz
    # ============================================================

    charges_atoms = read(str(CHARGES_XYZ))

    charges_raw = []
    symbols_raw = []

    with open(CHARGES_XYZ) as f:

        lines = f.readlines()[2:]

        for line in lines:

            fields = line.split()

            symbols_raw.append(fields[0])

            # last column = charge
            charges_raw.append(float(fields[-1]))

    symbols_raw = np.array(symbols_raw)
    charges_raw = np.array(charges_raw)

    # ============================================================
    # Validate CHARGES_XYZ vs TARGET_XYZ ordering
    # ============================================================

    assert len(charges_atoms) == len(target_raw)

    geom_diff = np.linalg.norm(
        charges_atoms.positions
        - target_raw.positions,
        axis=1,
    )

    assert geom_diff.max() < 1e-5, (
        "CHARGES_XYZ and TARGET_XYZ "
        "do not share identical atom ordering."
    )

    # ============================================================
    # Apply permutation
    # ============================================================

    symbols_perm = symbols_raw[perm]
    charges_perm = charges_raw[perm]

    # Consistent Si -> S mapping
    symbols_perm = np.where(
        symbols_perm == "Si",
        "S",
        symbols_perm,
    )

    # ============================================================
    # Validate permuted geometry
    # ============================================================

    charges_reordered = charges_atoms[perm]

    charges_reordered.set_chemical_symbols(
        reference_atoms.get_chemical_symbols()
    )

    perm_geom_diff = np.linalg.norm(
        charges_reordered.positions
        - target_prmtop_order.positions,
        axis=1,
    )

    assert perm_geom_diff.max() < 1e-5, (
        "Permutation inconsistency detected."
    )

    # ============================================================
    # Validate symbols after permutation
    # ============================================================

    assert np.array_equal(
        symbols_perm,
        reference_symbols
    ), "Chemical symbol mismatch after permutation."

    # ============================================================
    # Update charges
    # ============================================================

    calc_retargeted.update_charges(
        range(len(charges_perm)),
        charges_perm,
    )

    return calc_retargeted


def system_factory():
    """
    Return fresh unconstrained (system, topology).
    """

    _, system, topology = from_amber(
        str(PRMTOP),
        str(INPCRD),
        nonbonded_method='NoCutoff',
        constraints=None,
    )

    return system, topology



calc = make_calc_retargeted()



atoms = calc.loaded_atoms


calc.solvate(
    box_size=55.0,
    solvent="THF",
    n_molecules=1234,
    work_dir="thf_solvation",
)

v_corr_params = np.array([+18.7668, +1.0209, -35.4731]) # or hard-code: np.array([-21.94, -11.93, -26.13])
correction = FourierTorsionCorrection(
    atom_indices=[31, 28, 6, 2],
    n_harmonics=2,
    cosine_only=True,
)
calc.add_correction(correction, v_corr_params)


print("Box solvated")

print(f"Minimization has started")

calc.minimize(tolerance=10.0, max_iterations=500)
calc.save_prmtop('nvt_pbe_min')

print(f"Minimization has finished")

print(f"NPT simulation has started")

# 1. NPT equilibration (box relaxes to equilibrium density)
calc.run_npt(   
    steps=2_000_000,   # 2 ns with 1 fs timestep
    timestep=1.0, # fs
    dcd_file='nvt_pbe_npt.dcd',
    temperature=300,
    friction=1.0,  # ps^-1  
    log_file='nvt_pbe_npt.log',
    log_interval=2000,
    dcd_interval=2000,
)


calc.save_prmtop('nvt_pbe_npt')  # → .prmtop + .inpcrd
calc.save_system_xml('nvt_pbe_npt.xml', 'nvt_pbe_npt.pdb')

print(f"NPT simulation has finished")

print(f"NVT simulation has started")


calc.run_nvt(   
    steps=100_000_000,   # 2 ns with 1 fs timestep
    timestep=1.0, # fs
    dcd_file='nvt_pbe_nvt.dcd',
    temperature=300,
    friction=1.0,  # ps^-1  
    log_file='nvt_pbe_nvt.log',
    log_interval=10000,
    dcd_interval=10000,
)

calc.save_prmtop('nvt_pbe_nvt')  # → .prmtop + .inpcrd
calc.save_system_xml('nvt_pbe_nvt.xml', 'nvt_pbe_nvt.pdb')

print(f"NVT simulation has finished")


In [ ]:
time = np.loadtxt('nvt_pbe_nvt.log', usecols=1, skiprows=1, delimiter=',')
kinenergy = np.loadtxt('nvt_pbe_nvt.log', usecols=3, skiprows=1, delimiter=',')
potenergy = np.loadtxt('nvt_pbe_nvt.log', usecols=2, skiprows=1, delimiter=',')
totenergy = kinenergy + potenergy
temperature = np.loadtxt('nvt_pbe_nvt.log', usecols=4, skiprows=1, delimiter=',')


time = time / 1000000  # Convert from fs to ns

# --- Plot ---
fig, ax = plt.subplots(2, 1 , figsize=(10, 12))
ax[0].plot(time, totenergy, linewidth=2, color='black', label='Total Energy (kJ/mol)')
ax[1].plot(time, temperature, linewidth=2, color='red', label='Temperature (K)')

# --- Labels and style ---

for a in ax:
    a.set_xlabel("Time (ns)", fontsize=12)

ax[0].set_ylabel("Total Energy (kJ/mol)", fontsize=12)
ax[1].set_ylabel("Temperature (K)", fontsize=12)
#ax.set_xlim(0, 12)
#ax.set_ylim(0.0, 1.1) 
ax[0].legend(loc='upper left', fontsize=10)
ax[1].legend(loc='upper left', fontsize=10)
plt.tight_layout()
#fig.savefig("eq_npt_energy.png", dpi=400, bbox_inches="tight")
plt.show()

In [ ]:
DIH_IDX = [31, 28, 6, 2]
# angles_gaff = sample_dihedral_manual('nvt_pbe_min.prmtop', 'nvt_pbe_npt.dcd', DIH_IDX)
angles_corr2 = sample_dihedral_manual('nvt_pbe_min.prmtop', 'nvt_pbe_nvt.dcd', DIH_IDX)

for i in range(len(angles_corr2)):
    if angles_corr2[i] < 0:
        angles_corr2[i] = angles_corr2[i] + 360
    else:
        angles_corr2[i] = angles_corr2[i]


for i in range(len(angles_corr2)):
    if 0 <= angles_corr2[i] <= 90:
        pass
    elif 90 < angles_corr2[i] <= 180:
        angles_corr2[i] = 180 - angles_corr2[i]
    elif 180 < angles_corr2[i] <= 270:
        angles_corr2[i] = angles_corr2[i] - 180
    elif 270 < angles_corr2[i] <= 360:
        angles_corr2[i] = 360 - angles_corr2[i]

# bins = np.arange(-180, 184, 4)
bins = np.linspace(0, 90, 19)
fig, ax = plt.subplots(figsize=(9, 6))
# ax.hist(angles_gaff, bins=bins, density=True, alpha=0.6, label='GAFF')
ax.hist(angles_corr2, bins=bins, density=True, alpha=0.6) 
ax.set_xlabel('Dihedral angle (deg)')
ax.set_xticks(np.linspace(0, 90, 19))
ax.set_ylabel('Probability density')
# ax.set_ylim(0,0.008)
ax.set_title('Dihedral angle distribution in vacuum. 2 ns NVT')  
ax.legend()
fig.tight_layout()
# plt.savefig("ff_potentialfitting_binscharges.png", dpi=400, bbox_inches="tight")
plt.show()

Calculation of the Potential of Mean Force

In [ ]:
kb = 8.617333262e-5 # Boltzmann constant in eV/K
T = 300 # Temperature in K
#pmf = -kb * T * np.log(np.histogram(angles_corr2, bins=np.arange(0, 360, 4), density=True)[0])
bins=np.arange(0, 92, 2)
pmf = -kb * T * np.log(np.histogram(angles_corr2, bins=bins, density=True)[0])
pmf -= np.min(pmf)  # Shift PMF to have minimum at 0 eV 
pmf = pmf * 1000 # Convert from eV to meV

## 5-Nonequilibrium MD

For the nonequilibrium MD we mimic the situation where a portion of the population has been exciten by removing those trayectories which start in a certain dihedral angle range. Then we divide the trajectory in chunks with delay to capture all the posible statistical information of the trajectory without performing more simulations. 

In [ ]:
DIH_IDX = [31, 28, 6, 2]
angles_corr2 = sample_dihedral_manual('ogtraj100ns.prmtop', 'ogtraj100ns.dcd', DIH_IDX)



for i in range(len(angles_corr2)):
    if angles_corr2[i] < 0:
        angles_corr2[i] = angles_corr2[i] + 360
    else:
        angles_corr2[i] = angles_corr2[i]


for i in range(len(angles_corr2)):
    if 0 <= angles_corr2[i] <= 90:
        angles_corr2[i] = angles_corr2[i]
    elif 90 < angles_corr2[i] <= 180:
        angles_corr2[i] = 180 - angles_corr2[i]
    elif 180 < angles_corr2[i] <= 270:
        angles_corr2[i] = angles_corr2[i] - 180
    elif 270 < angles_corr2[i] <= 360:
        angles_corr2[i] = 360 - angles_corr2[i]

meananglefull = np.mean(angles_corr2)
print(meananglefull)

time = np.loadtxt('ogtraj100ns.log', usecols=1, skiprows=1, delimiter=',')
time = time / 1000000  # Convert from fs to ns

window_size = 90   # frames por chunk
delay = 10          # desplazamiento entre chunks

final_angles = []
final_time = []
k = 0


for i in range(0, len(angles_corr2) - window_size + 1, delay):

    chunk_angles = angles_corr2[i:i+window_size]
    chunk_time = time[i:i+window_size]

    if chunk_angles[0] <= 20:
        k+=1
        continue

    final_angles.append(chunk_angles)
    final_time.append(chunk_time)

final_angles = np.array(final_angles)
final_time = np.array(final_time)

print("Skipped chunks:", k)
print("Frames por chunk:", window_size)
print("Delay:", delay)
print("Total number of chunks:",
      (len(angles_corr2) - window_size) // delay + 1)
total_chunks = (len(angles_corr2) - window_size) // delay + 1
print(
    f"Chunks with a dihedral angle higher than 20 deg: "
    f"{len(final_angles)} ({len(final_angles) / total_chunks * 100:.1f}%)"
)


A parameter that we measure from the nonequilibrium MD is the time-dependent mean. The script averages the first frame of all the chunks.

In [ ]:

oganglesdelay2 = final_angles
print(oganglesdelay2.shape)
meanangle=[]
for i in range(0,len(oganglesdelay2[1])):
    col = np.mean(oganglesdelay2[:,i])
    meanangle.append(col)
print(len(meanangle))
meanangle = np.array(meanangle) 
temptime = range(0,900,10)   
fig, ax = plt.subplots(1, 1, figsize=(9, 6))
ax.plot(temptime, meanangle, marker='o', label='Mean dihedral angle of all chunks in each frame')
# ax.hist(meanangle, edgecolor='black', linewidth=1.0, density=True)
ax.hlines(meananglefull, 0, 900, linestyles='--', colors='black', label='Mean dihedral angle of the whole trajectory')
ax.set_xlabel('Time (ps)')
ax.set_xticks(range(0, 1000, 100))
# ax.set_yticks(range(0,90, 10))
ax.set_ylabel('Dihedral angle (º)')
ax.set_title('Average dihedral angle of all chunks')
ax.legend()

fig.tight_layout()
plt.savefig("avgangle.png", dpi=400, bbox_inches="tight")
plt.show()

This script compares the nonequilibrium dihedral angle distribution and the dihedral angle distribution after reaching the equilibrium again

In [ ]:
oganglesdelay = final_angles
firstangle=[]
lastangle=[]
for i in range(len(oganglesdelay)):
    firsttemp = oganglesdelay[i,0]
    firstangle.append(firsttemp)
    lasttemp = oganglesdelay[i,-1]
    lastangle.append(lasttemp)
    # print(meanangle)


firstangle = np.array(firstangle)  
lastangle = np.array(lastangle)
bins = np.arange(0,95,5)
fig, ax = plt.subplots(1, 1, figsize=(9, 6))
ax.hist(angles_corr2, bins=bins, density=True, alpha=0.6, label='Full trajectory', )
ax.hist(firstangle, bins=bins, density=True, alpha=0.6, label='t = 0 ps', )
# ax.plot(meanangle, marker='o',)
# ax.set_xlabel('Time x 10 (ps)')
ax.set_xlabel('Dihedral angle (º)')
ax.set_xticks(range(0, 95, 10))
# ax.set_xticks(range(0, 200, 10))
# ax.set_yticks(range(0,90, 10))
ax.set_ylabel('Probability density')
# ax.set_ylabel('Dihedral angle (º)')
ax.set_title(f'Non-equilibrum MD')
ax.legend(loc='upper left', fontsize=10)
fig.tight_layout()
plt.savefig("noneqmd.png", dpi=400, bbox_inches="tight")
plt.show()